# Audit des données

Objectif : vérifier la disponibilité et la qualité des données
avant de comparer les portefeuilles.

Première étape : collecte pilote de SPY, IEF, GLD et BIL
sur la période 2019–2023.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf
from IPython.display import display

current_dir = Path.cwd().resolve()

project_dir = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "config.py").is_file()
        and (folder / "src").is_dir()
    ),
    None,
)

if project_dir is None:
    raise FileNotFoundError("Impossible de retrouver le dossier du projet.")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from config import TICKERS, PILOT_START, PILOT_DOWNLOAD_END

print("Projet :", project_dir)
print("Python :", sys.executable)
print("Version pandas :", pd.__version__)
print("Version yfinance :", yf.__version__)
print("ETF :", TICKERS)

Projet : /home/onyxia/Projet-Python-ENSAE
Python : /opt/python/bin/python
Version pandas : 3.0.6
Version yfinance : 1.7.0
ETF : ['SPY', 'IEF', 'GLD', 'BIL']


In [3]:
# Décembre 2018
download_start = (
    pd.Timestamp(PILOT_START) - pd.DateOffset(months=1)
).strftime("%Y-%m-%d")

spy_test = yf.download(
    "SPY",
    start=download_start,
    end=PILOT_DOWNLOAD_END,
    interval="1d",
    auto_adjust=False,
    actions=True,
    repair=False,
    keepna=True,
    threads=False,
    progress=False,
    multi_level_index=False,
    timeout=30,
)

if spy_test is None or spy_test.empty:
    raise RuntimeError(
        "Yahoo Finance n'a renvoyé aucune donnée pour SPY. "
        "Examiner les messages affichés avant de poursuivre."
    )

print("Dimensions :", spy_test.shape)
print("Première date :", spy_test.index.min())
print("Dernière date :", spy_test.index.max())

display(spy_test.head())

Dimensions : (1277, 9)
Première date : 2018-12-03 00:00:00
Dernière date : 2023-12-29 00:00:00


,Adj Close,Capital Gains,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,,
2018-12-03,247.790222,0.0,279.299988,0.0,280.399994,277.510010,280.279999,0.0,103176300
2018-12-04,239.761230,0.0,270.250000,0.0,278.850006,269.899994,278.369995,0.0,177986000
2018-12-06,239.397446,0.0,269.839996,0.0,269.970001,262.440002,265.920013,0.0,204185400
2018-12-07,233.834839,0.0,263.570007,0.0,271.220001,262.630005,269.459991,0.0,161018900
2018-12-10,234.278442,0.0,264.070007,0.0,265.160004,258.619995,263.369995,0.0,151445900


In [5]:
expected_columns = [
    "Close",
    "Adj Close",
    "Dividends",
    "Stock Splits",
]

missing_columns = set(expected_columns) - set(spy_test.columns)

if missing_columns:
    raise ValueError(f"Colonnes absentes : {missing_columns}")

display(spy_test[expected_columns].head())

display(
    spy_test[expected_columns]
    .isna()
    .sum()
    .rename("valeurs_manquantes")
    .to_frame()
)

,Close,Adj Close,Dividends,Stock Splits
Date,,,,
2018-12-03,279.299988,247.790222,0.0,0.0
2018-12-04,270.250000,239.761230,0.0,0.0
2018-12-06,269.839996,239.397446,0.0,0.0
2018-12-07,263.570007,233.834839,0.0,0.0
2018-12-10,264.070007,234.278442,0.0,0.0


,valeurs_manquantes
Close,0
Adj Close,0
Dividends,0
Stock Splits,0


In [6]:
distributions = spy_test.loc[
    spy_test["Dividends"].fillna(0).ne(0),
    ["Close", "Adj Close", "Dividends"],
]

print("Nombre de distributions observées :", len(distributions))
display(distributions.head())

Nombre de distributions observées : 21


,Close,Adj Close,Dividends
Date,,,
2018-12-21,240.699997,214.791977,1.435
2019-03-15,281.309998,252.136566,1.233
2019-06-21,294.000000,264.792175,1.432
2019-09-20,298.279999,269.887573,1.384
2019-12-20,320.730011,291.627472,1.570
